In [1]:
from llm_api import generate_new_heuristic
from tsp_sandbox import load_tsp_data_from_csv, evaluate_heuristic

# --- 初始启发式算法（最近邻，作为基线） ---
INITIAL_CODE = """
def select_next_node(distance_matrix, current_node, unvisited_nodes):
    min_dist = float('inf')
    best_node = unvisited_nodes[0]
    for node in unvisited_nodes:
        if distance_matrix[current_node][node] < min_dist:
            min_dist = distance_matrix[current_node][node]
            best_node = node
    return best_node
"""

def main():
    print("🚀 启动 Thoughts-augmented FunSearch...")
    # 我们先测试 5 个实例，跑得快。如果想测更多，可以改成 25
    dataset = load_tsp_data_from_csv("tsp_instances_dataset.csv", num_instances=25)
    
    if not dataset:
        return
        
    # 1. 评估初始基线，拿到各关卡的“标尺”距离
    # 注意：此时因为没传 baseline_distances，返回的是 (平均绝对距离, 各关卡距离列表)
    best_avg_raw_distance, baseline_distances, best_paths = evaluate_heuristic(INITIAL_CODE, dataset)
    best_code = INITIAL_CODE
    
    print(f"\n📍 初始基线算法在 {len(dataset)} 个地图上的成绩如下 (平均距离: {best_avg_raw_distance:.2f}):")
    for i, dist in enumerate(baseline_distances):
        print(f"   - 地图 {i+1} ({dataset[i]['num_cities']}城): {dist:.2f}")
    print("-" * 50 + "\n")
    
    # 现在的最佳得分统一定义为 Ratio（相对于基线的比率）。基线自己的 Ratio 就是 1.0。越小越好！
    best_score_ratio = 1.0
    
    ITERATIONS = 30 
    for i in range(ITERATIONS):
        print(f"========== 迭代 {i+1}/{ITERATIONS} ==========")
        
        # ⚠️ 【强制洗脑版 Prompt】：封杀 Look-ahead，逼迫使用其他几何策略
        # ⚠️ 【微调演进版 Prompt】：禁止彻底重写，强制在距离上叠加软惩罚权重
        prompt = (
            "You are an expert algorithm scientist optimizing a heuristic for the Traveling Salesman Problem (TSP).\n"
            f"Current best Relative Score Ratio: {best_score_ratio:.4f} (lower is better. 1.0 is the Nearest Neighbor baseline).\n\n"
            "CURRENT BEST CODE TO IMPROVE:\n"
            f"```python\n{best_code}\n```\n\n"
            "INSTRUCTIONS:\n"
            "1. [Design Thought]: Analyze how to improve the current code.\n"
            "   - WARNING: The current greedy approach is a VERY STRONG baseline. DO NOT completely rewrite the logic. DO NOT use '1-step Look-ahead' (it fails on these maps).\n"
            "   - INSTEAD, implement a 'Hybrid Weighting' strategy.\n"
            "   - Compute a score for each candidate: `score = distance_to_candidate + alpha * penalty_term`.\n"
            "   - The `penalty_term` can be an isolation metric, or the candidate's distance to the centroid of remaining nodes (using a negative alpha to favor visiting outliers early). \n"
            "   - Keep the weight `alpha` relatively small (e.g., 0.1 to 0.5) so it gently guides the greedy choice rather than destroying it.\n"
            "2. [Python Code]: Implement your adjusted scoring mechanism.\n"
            "   - Evaluate all unvisited nodes and pick the one with the MINIMUM custom score.\n"
            "   - CRITICAL: Handle `len(unvisited_nodes) == 1` properly.\n"
            "   - Return ONLY valid Python code inside ```python ``` blocks."
        )
        
        print("🧠 正在请求 DeepSeek 进行思考与编码...")
        full_response, new_code = generate_new_heuristic(prompt)
        
        if not new_code:
            print("❌ 未能生成有效代码，跳过...")
            continue
            
        print("\n--- 💡 DeepSeek 的设计思路 (Design Thought) ---")
        thought = full_response.split("```python")[0].strip()
        print(thought[:500] + "...\n(思路已截断)")
        print("----------------------------------------------\n")
            
        print("⚙️ 评估新代码...")
        # 2. 评估新代码（传入 baseline_distances 作为对比标尺）
        # 此时返回的是 (平均比率, 各关卡距离列表)
        score_ratio, new_distances, new_paths = evaluate_heuristic(new_code, dataset, baseline_distances=baseline_distances)
    
        if score_ratio == float('inf') or not new_distances:
            print("💥 新代码运行失败或逻辑不合法 (可能遇到了边界Bug)。")
        elif score_ratio < best_score_ratio:
            print(f"🎉 进化成功！发现了更优的逻辑！")
            print(f"📉 综合提升比例: {(1 - score_ratio)*100:.2f}% (综合得分: {score_ratio:.4f})")
            
            # 打印每一关的具体成绩对比！
            print("📊 详细战报：")
            for idx, ndist in enumerate(new_distances):
                bdist = baseline_distances[idx]
                print(f"   - 地图 {idx+1}: {bdist:.2f} -> {ndist:.2f} (比率: {(ndist/bdist):.3f})")
                
            best_score_ratio = score_ratio
            best_code = new_code
            best_paths = new_paths # 更新最优路径
            
            # 1. 【保存算法代码】：存为 .py，这是你的“最强大脑”
            with open("best_tsp_heuristic.py", "w", encoding="utf-8") as f:
                f.write(best_code)
            
            # 2. 【保存行走路径】：存为 .json，这是你的“完美脚印”
            import json
            with open("best_paths.json", "w", encoding="utf-8") as f:
                json.dump(best_paths, f)
                
            print(f"💾 成果已保存：代码 -> best_tsp_heuristic.py, 路径数据 -> best_paths.json")
        else:
            print(f"📉 进化失败。新算法没有改进。当前综合得分: {score_ratio:.4f} (历史最佳: {best_score_ratio:.4f})")
        print("\n")

    print("🏁 ========== 搜索结束 ==========")
    print(f"🏆 发现的最优综合得分 (Ratio): {best_score_ratio:.4f}")
    print("📜 最优代码已保存在 best_tsp_heuristic.py 中。最后版本如下:")
    print(best_code)

if __name__ == "__main__":
    main()

🚀 启动 Thoughts-augmented FunSearch...

📍 初始基线算法在 25 个地图上的成绩如下 (平均距离: 902.16):
   - 地图 1 (54城): 653.18
   - 地图 2 (106城): 959.42
   - 地图 3 (29城): 616.76
   - 地图 4 (41城): 588.08
   - 地图 5 (140城): 1047.35
   - 地图 6 (32城): 542.99
   - 地图 7 (86城): 904.74
   - 地图 8 (109城): 1100.21
   - 地图 9 (70城): 787.03
   - 地图 10 (112城): 993.37
   - 地图 11 (112城): 964.36
   - 地图 12 (135城): 1129.41
   - 地图 13 (110城): 924.59
   - 地图 14 (75城): 793.51
   - 地图 15 (121城): 1110.03
   - 地图 16 (83城): 862.08
   - 地图 17 (114城): 1029.89
   - 地图 18 (114城): 1028.77
   - 地图 19 (27城): 595.14
   - 地图 20 (143城): 1142.91
   - 地图 21 (142城): 1108.12
   - 地图 22 (130城): 1097.82
   - 地图 23 (34城): 551.89
   - 地图 24 (128城): 1033.58
   - 地图 25 (107城): 988.76
--------------------------------------------------

========== 迭代 1/30 ==========
🧠 正在请求 DeepSeek 进行思考与编码...

--- 💡 DeepSeek 的设计思路 (Design Thought) ---
Here's an improved version that adds a hybrid weighting strategy while maintaining the core greedy approach. The penalty term cons

<string>:29: RuntimeWarning: invalid value encountered in subtract
c:\Users\14480\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


📉 进化失败。新算法没有改进。当前综合得分: 5.3096 (历史最佳: 0.9582)


========== 迭代 19/30 ==========
🧠 正在请求 DeepSeek 进行思考与编码...

--- 💡 DeepSeek 的设计思路 (Design Thought) ---
I'll analyze the current code and propose improvements while maintaining its core structure. The current approach already uses a hybrid weighting strategy with distance and isolation metrics, but we can refine the components for better balance and performance.

Key observations:
1. The isolation score combines centroid deviation and minimum distance to others - a good approach
2. Current alpha scaling is complex but could be simplified while maintaining effectiveness
3. The centroid is computed ...
(思路已截断)
----------------------------------------------

⚙️ 评估新代码...
📉 进化失败。新算法没有改进。当前综合得分: 1.0092 (历史最佳: 0.9582)


========== 迭代 20/30 ==========
🧠 正在请求 DeepSeek 进行思考与编码...

--- 💡 DeepSeek 的设计思路 (Design Thought) ---
Here's my improved version focusing on refining the hybrid weighting strategy while maintaining the core logic. Key improvements inc

<string>:18: RuntimeWarning: invalid value encountered in multiply


📉 进化失败。新算法没有改进。当前综合得分: 5.3096 (历史最佳: 0.9582)


========== 迭代 21/30 ==========
🧠 正在请求 DeepSeek 进行思考与编码...

--- 💡 DeepSeek 的设计思路 (Design Thought) ---
I'll analyze the current code and propose improvements while maintaining its core structure. The current approach uses a weighted combination of centroid deviation and isolation distance, but we can refine the weighting strategy for better performance.

Key observations:
1. The centroid deviation component currently uses only distance from node 0, but could benefit from considering the current_node's perspective.
2. The isolation distance calculation is computationally expensive with nested loop...
(思路已截断)
----------------------------------------------

⚙️ 评估新代码...
📉 进化失败。新算法没有改进。当前综合得分: 0.9779 (历史最佳: 0.9582)


========== 迭代 22/30 ==========
🧠 正在请求 DeepSeek 进行思考与编码...

--- 💡 DeepSeek 的设计思路 (Design Thought) ---
I'll analyze the current approach and suggest improvements while maintaining the core structure. The current code already implements